# Exploring masks and offsets and all movement operations

This is a document to try to understand masks and offsets, and other ways in which natural movement operations can create unusual situations.

Conclusions:

- mask: every dimension has a mask `(start, end)` that indicates the range of allowed values `i` for the index in that dimension (`start <= i < end`). Values outside of the mask represent a zero (I think).
- offset: there is only a single offset; you start the strides from there

Mops = movement operations:

- shrink: this creates an offset, and changes the shape
- pad: adds values; this creates masks. If you pad a non-zero value, then you just need to create new memory.
- expand: this is broadcasting; creates 0 strides
- permute: permutes the shape and strides
- reshape: this adds the canonical view corresponding to the shape to the shapetracker
- stride: multiplies strides by a factor; changes shape; if a multiplication factor is negative, introduces an offset

These mops all act on the shapetracker (see the `mops` dict in `shapetracker.py`), and most of these are relatively straightforwardly passed on to the views.

In [2]:
import os
os.environ['CACHELEVEL'] = '0'

from tinygrad import Tensor
from tinygrad.shape.shapetracker import ShapeTracker
from tinygrad.shape.view import View, merge_dims, unravel
from tinygrad.ops import UOp

First, a simple example of where two views can not be merged.

Explanation: take a (2,3) matrix, transpose it, and flatten it. Because of the transposition, the data is intermingled, and that can not be expressed in a rank 1 matrix.


In [8]:
simple_matrix = Tensor.arange(6).reshape(2,3)
simple_matrix.permute((1, 0)).reshape(6).lazydata.st.simplify().views

(View(shape=(3, 2), strides=(1, 3), offset=0, mask=None, contiguous=False),
 View(shape=(6,), strides=(1,), offset=0, mask=None, contiguous=True))

Now let's investigate how we get offsets and masks.

## Example 1: Tensor.eye

Pictorially, to create e.g. Tensor.eye(3), this is what happens:

```
1       1 0 0 0       1 0 0 0 1 0 0 0 1 0 0 0        1 0 0 0 1 0 0 0 1            1 0 0
1   ->  1 0 0 0  ->                              ->                     ->        0 1 0
1   pad 1 0 0 0 flatten                         shrink                 reshape    0 0 1
```


So in words: if you were to flatten `Tensor.eye(3)`, you'd notice that there are 3 zeroes between every pair of ones, and you can create that by padding and shrinking.

In [10]:
Tensor.eye(3).lazydata.st.simplify().views

(View(shape=(3, 4), strides=(0, 0), offset=0, mask=((0, 3), (0, 1)), contiguous=False),
 View(shape=(3, 3), strides=(3, 1), offset=0, mask=None, contiguous=True))

In [11]:
Tensor.eye(4).lazydata.st.simplify().views

(View(shape=(4, 5), strides=(0, 0), offset=0, mask=((0, 4), (0, 1)), contiguous=False),
 View(shape=(4, 4), strides=(4, 1), offset=0, mask=None, contiguous=True))

In [12]:
Tensor.ones((5,1)).lazydata.st.simplify().views

(View(shape=(5, 1), strides=(0, 0), offset=0, mask=None, contiguous=False),)

By adding padding, we introduce a mask. The mask tells us for each dimension what range of index values are allowed in that dimension (left-inclusive, right-open).


In [13]:
Tensor.ones((5,1)).pad((None,(0,5))).lazydata.st.simplify().views

(View(shape=(5, 6), strides=(0, 0), offset=0, mask=((0, 5), (0, 1)), contiguous=False),)

In [14]:
Tensor.ones((5,1)).pad((None,(0,5))).flatten().shrink(((0,25),)).lazydata.st.simplify().views

(View(shape=(5, 6), strides=(0, 0), offset=0, mask=((0, 5), (0, 1)), contiguous=False),
 View(shape=(25,), strides=(1,), offset=0, mask=None, contiguous=True))

The general function:

In [15]:
n = 3
x = Tensor.ones((n,1)).pad((None,(0,n))).flatten().shrink(((0,n*n),)).reshape(n,n)

### What this teaches us about masks

- Mask: for every dimension, the range of index values (`(start, end)`) that are allowed. Values outside of this mask are assumed to be zero. If you pad with a value other than zero, you just create new data.


If you pad with a value other than zero, tinygrad is forced to realize it and just create new memory (I think):

In [18]:
Tensor.ones((5,1)).pad((None,(0,5)), value=2).lazydata.st.simplify().views

(View(shape=(5, 6), strides=(6, 1), offset=0, mask=None, contiguous=True),)

In [9]:
Tensor.ones((5,1))[3,0].item()

1.0

In [14]:
ones_padded = Tensor.ones((5,1)).pad((None,(1,2)), value=0)
ones_padded.lazydata.st.simplify().views, ones_padded[3,0].item(), ones_padded[3,1].item()

((View(shape=(5, 4), strides=(0, 0), offset=0, mask=((0, 5), (1, 2)), contiguous=False),),
 0.0,
 1.0)

## Example 2: shrinking to get offsets

If you shrink `[0 1 2 3 4 5 6 7 8 9]` to `[3 4 5]`, you can express that with an offset of 3 and a shape of `(3,)`.


In [23]:
x = Tensor.arange(10).shrink(((3,6),))
x.numpy(), x.lazydata.st.simplify().views

(array([3, 4, 5], dtype=int32),
 (View(shape=(3,), strides=(1,), offset=3, mask=None, contiguous=False),))

Can't you have several offsets, one for each dimension? No, if you shrink several dimensions, you can just express that with a single offset.

Note that in the example below, the offset is `3 * 10 + 3 * 1 = 33` (where the 10 and 1 come from the strides).

In [24]:
x = Tensor.arange(100).reshape(10,10).shrink(((3,6),(3,6)))
x.numpy(), x.lazydata.st.simplify().views


(array([[33, 34, 35],
        [43, 44, 45],
        [53, 54, 55]], dtype=int32),
 (View(shape=(3, 3), strides=(10, 1), offset=33, mask=None, contiguous=False),))

A contrived example: if you shrink to 2 elements, you should always be able to find an appropriate stride:


In [30]:
x = Tensor.arange(10).reshape(2,5).permute((1,0)).flatten().shrink(((3, 5),))
x.numpy(), x.lazydata.st.simplify().views


(array([6, 2], dtype=int32),
 (View(shape=(2,), strides=(-4,), offset=6, mask=None, contiguous=False),))

But shrinking to three elements doesn't work:

In [31]:
x = Tensor.arange(10).reshape(2,5).permute((1,0)).flatten().shrink(((3, 6),))
x.numpy(), x.lazydata.st.simplify().views


(array([6, 2, 7], dtype=int32),
 (View(shape=(5, 2), strides=(1, 5), offset=0, mask=None, contiguous=False),
  View(shape=(3,), strides=(1,), offset=3, mask=None, contiguous=False)))

## Example 3: Expanding: gets you zero strides

Pad adds some values, whereas expand is broadcasting. So you can not expand if the dimension has shape > 1.

In [34]:
x = Tensor.ones((5,)).expand((5,5))
x.numpy(), x.lazydata.st.simplify().views



(array([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]], dtype=float32),
 (View(shape=(5, 5), strides=(0, 0), offset=0, mask=None, contiguous=False),))

In [36]:
x = Tensor.arange((5)).expand((4,5))
x.numpy(), x.lazydata.st.simplify().views



(array([[0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4]], dtype=int32),
 (View(shape=(4, 5), strides=(0, 1), offset=0, mask=None, contiguous=False),))

## Example 4: reshape

Adds the canonical view corresponding to the shape to the views in the shapetracker.

The canonical view is just the row major strides corresponding to the shape. Example of canonical view: for shape `(5, 3, 2)`, it is `(3 * 2, 2, 1)`.

Implementation: adds a `UOps.RESHAPE`, which ultimately triggers `ShapeTracker.reshape`

In [37]:
x = Tensor.arange(10).reshape(2,5).permute((1,0)).reshape(10)
x.numpy(), x.lazydata.st.simplify().views

(array([0, 5, 1, 6, 2, 7, 3, 8, 4, 9], dtype=int32),
 (View(shape=(5, 2), strides=(1, 5), offset=0, mask=None, contiguous=False),
  View(shape=(10,), strides=(1,), offset=0, mask=None, contiguous=True)))

## Example 5: stride

`Ops.stride` is also one of the "movement ops"/"mops". It multiplies each stride by some factor.

Strangely, this is not an operation on tensors, but only on `UOp`s.

In [40]:
x = Tensor.arange(10).lazydata.stride((2,5))

AttributeError: 'UOp' object has no attribute 'numpy'